In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

2.2.2


In [3]:
# =========================
# Pipeline: CDO mergetime per variable -> final xarray merge
# =========================
import glob
import xarray as xr

# Paths
base_path    = "/work/uc1275/u301827/02_MSE/full_midlatitude/raw"
scratch_path = "/scratch/u/u301827/full_midlatitude/"
out_file     = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(os.path.dirname(out_file), exist_ok=True)

# Variables (folders under base_path)
vars_to_merge = [
    "tasmax",
    "q",
    "t",
    "z",
    "sp",
    "2d",
    "blh",
    "swvl1",
]

# Chunking for low RAM when opening the merged products
CHUNKS = {"time": 50}  # tune if you want (e.g. 10, 30, 100)


def cdo_mergetime(files, out_nc):
    """
    Merge a list of monthly files into a single time-series file using CDO.
    """
    cdo = Cdo()
    
    if not files:
        return None

    files = sorted(files)

    # CDO expects a whitespace-separated list of files
    in_str = " ".join(files)

    # Overwrite output if exists
    if os.path.exists(out_nc):
        os.remove(out_nc)

    # -O overwrite, -f nc output netcdf
    cdo.mergetime(input=in_str, output=out_nc, options="-O -f nc")
    return out_nc


def build_merged_filepaths_for_var(var):
    """
    Return paths to intermediate (scratch) merged files for:
      - daily merged file
      - at_tasmax merged file (if exists)
    tasmax is special: only daily exists, filename pattern differs.
    """
    var_path = os.path.join(base_path, var)
    if not os.path.isdir(var_path):
        print(f"[WARN] Missing folder: {var_path}")
        return None, None

    if var == "tasmax":
        daily_files = glob.glob(os.path.join(var_path, "tasmax_*_midlatitudes.nc"))
        daily_out   = os.path.join(scratch_path, "tasmax_merged.nc")
        return (sorted(daily_files), daily_out), ([], None)

    daily_files = glob.glob(os.path.join(var_path, f"{var}_????-??.nc"))
    at_files    = glob.glob(os.path.join(var_path, f"{var}_????-??_at_tasmax.nc"))

    daily_out = os.path.join(scratch_path, f"{var}_merged.nc")
    at_out    = os.path.join(scratch_path, f"{var}_merged_at_tasmax.nc")

    return (sorted(daily_files), daily_out), (sorted(at_files), at_out)


# =========================
# 1) CDO: mergetime per variable (and *_at_tasmax if present)
# =========================
merged_products = []  # list of (path, kind, var)

for var in vars_to_merge:
    cdo = Cdo()
    
    print(f"[INFO] CDO mergetime: {var}")

    (daily_files, daily_out), (at_files, at_out) = build_merged_filepaths_for_var(var)

    if daily_files:
        outp = cdo_mergetime(daily_files, daily_out)
        if outp:
            merged_products.append((outp, "daily", var))
            print(f"  -> wrote {outp}")
    else:
        print(f"  [WARN] No daily files found for {var}")

    # only for non-tasmax vars typically
    if at_files:
        outp = cdo_mergetime(at_files, at_out)
        if outp:
            merged_products.append((outp, "at_tasmax", var))
            print(f"  -> wrote {outp}")
    else:
        # silent for vars that don't have it (e.g., many might not)
        pass


[INFO] CDO mergetime: tasmax
  -> wrote /scratch/u/u301827/full_midlatitude/tasmax_merged.nc
[INFO] CDO mergetime: q
  -> wrote /scratch/u/u301827/full_midlatitude/q_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/q_merged_at_tasmax.nc
[INFO] CDO mergetime: t
  -> wrote /scratch/u/u301827/full_midlatitude/t_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/t_merged_at_tasmax.nc
[INFO] CDO mergetime: z
  -> wrote /scratch/u/u301827/full_midlatitude/z_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/z_merged_at_tasmax.nc
[INFO] CDO mergetime: sp
  -> wrote /scratch/u/u301827/full_midlatitude/sp_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/sp_merged_at_tasmax.nc
[INFO] CDO mergetime: 2d
  -> wrote /scratch/u/u301827/full_midlatitude/2d_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/2d_merged_at_tasmax.nc
[INFO] CDO mergetime: blh
  -> wrote /scratch/u/u301827/full_midlatitude/blh_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/blh_mer

In [4]:
from cdo import Cdo
import os

cdo = Cdo()
#cdo.debug = True

# merged_products: list of (path, kind, var)
# out_file: final output netcdf
# scratch_path: place for temporary renamed copies

tmp_files_for_merge = []

for path, kind, var in merged_products:
    print(f"[INFO] Preparing for CDO merge: {path}")

    if var == "tasmax":
        # Keep as is
        tmp_files_for_merge.append(path)
        continue

    if kind == "daily":
        # Keep as-is (optionally select only that variable)
        # If you're sure the file only contains `var`, you can skip selname.
        tmp = os.path.join(scratch_path, f"keep_{var}.nc")
        cdo.selname(var, input=path, output=tmp)   # keeps only var
        tmp_files_for_merge.append(tmp)

    else:  # at_tasmax
        # Rename var -> var_at_tasmax, and keep only that renamed var
        tmp = os.path.join(scratch_path, f"{var}_at_tasmax.nc")
        cdo.chname(
            f"{var},{var}_at_tasmax",
            input=f"-selname,{var} {path}",
            output=tmp
        )
        tmp_files_for_merge.append(tmp)

# Merge all variables into one dataset
tmp_merged = os.path.join(scratch_path, "ds_merged_allvars.nc")
cdo.merge(input=" ".join(tmp_files_for_merge), output=tmp_merged)

# Sort by time (recommended)
cdo.sorttimestamp(input=tmp_merged, output=out_file)

print(f"[DONE] Final dataset written to: {out_file}")
print(f"[INFO] Intermediate merged files are in: {scratch_path}")


[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/tasmax_merged.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/q_merged.nc
Found operator:selname
# DEBUG - start =============================================================
CALL  :cdo -O -s -selname,q /scratch/u/u301827/full_midlatitude/q_merged.nc /scratch/u/u301827/full_midlatitude/keep_q.nc
STDOUT:
STDERR:
# DEBUG - end ===============================================================
RETURNCODE:0
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/q_merged_at_tasmax.nc
Found operator:chname
# DEBUG - start =============================================================
CALL  :cdo -O -s -chname,q,q_at_tasmax -selname,q /scratch/u/u301827/full_midlatitude/q_merged_at_tasmax.nc /scratch/u/u301827/full_midlatitude/q_at_tasmax.nc
STDOUT:
STDERR:
# DEBUG - end ===============================================================
RETURNCODE:0
[INFO] Preparing for CDO merge: /scratch/u/u30182

In [ ]:
import numpy as np
from scipy.special import lambertw
import metpy.calc as mpcalc
from metpy.units import units
import xarray as xr

def compute_LCL(T, qv, Rh, z, g=9.81, p0 = 1000*100):
    """
    Compute the LCL temperature and height from temperature, humidity, and geopotential.
    
    Parameters
    ----------
    T : array-like or xarray.DataArray
        Temperature at reference level (K)
    qv : array-like or xarray.DataArray
        Specific humidity at reference level (kg/kg)
    Rh : array-like or xarray.DataArray
        Relative humidity at reference level (0 to 1)
    z : array-like or xarray.DataArray
        Geopotential at reference level (m^2/s^2), will be divided by g to get height (m)
    g : float, optional
        Gravitational acceleration (m/s²), default is 9.81
    p : float, optional
        pressure level of input (default 1000 hPa)

    Returns
    -------
    TLCL : same shape as inputs
        Temperature at LCL (K)
    zLCL : same shape as inputs
        Height of LCL (m)
    """

    # Physical constants
    Ttr = 273.16  # Triple point temperature (K)
    E0v = 2.3740e6  # Latent heat of vaporization at Ttr (J/kg)
    cvl = 4119      # Specific heat liquid water (J/kg/K)
    cvv = 1418      # Specific heat water vapor (J/kg/K)
    Rv  = 461       # Gas constant for water vapor (J/kg/K)
    cpv = cvv + Rv
    Ra  = 287.04    # Gas constant for dry air (J/kg/K)
    cva = 719
    cpa = cva + Ra

    # Moist air specific gas constant and heat capacity
    Rm  = (1 - qv) * Ra + qv * Rv
    cpm = (1 - qv) * cpa + qv * cpv

    # Coefficients for LCL temperature equation
    a = cpm / Rm + (cvl - cpv) / Rv
    b = - (E0v - Ttr * (cvv - cvl)) / (Rv * T)
    c = b / a

    # Argument to Lambert W function
    RHcec = c * np.exp(c) * Rh**(1 / a)

    # Compute TLCL using -1 branch of Lambert W
    TLCL = T * c / lambertw(RHcec, k=-1).real

    # Compute zLCL from TLCL
    z = z / g  # convert geopotential to height
    zLCL = z + (cpm / g) * (T - TLCL)
    zLCL_old = z+cpm/g * (T-55-1/(1/(T-55)-np.log(Rh)/2840))
    
    pLCL =  p0 * (TLCL / T) ** (cpm / Rm)

    return TLCL, zLCL, pLCL


def compute_surface_mse(res):
    """
    Compute surface moist static energy (MSE) from 2m temperature, specific humidity, and surface geopotential.

    Parameters
    ----------
    res : dict
        Output of `open_era5_day`, containing:
          - res['datasets']['2t']: xarray.Dataset with 2m temperature (K)
          - res['datasets']['q'] : xarray.Dataset with specific humidity (kg/kg)
          - res['zs']            : xarray.DataArray with surface geopotential height (m)

    Returns
    -------
    mse_da : xarray.DataArray
        Surface moist static energy (J/kg) with the same coordinates as the 2m temperature field.
    """

    # --- Extract variables ---
    ds_2t = res["datasets"]["tasmax"][["tasmax","lat", "lon", "time"]]
    ds_q  = res["datasets"]["q"][["q","lat", "lon", "time"]]
    zs    = res["zs"]

    # Try to find the variable names automatically
    var_2t = list(ds_2t.data_vars)[0]
    var_q  = list(ds_q.data_vars)[0]

    T = ds_2t[var_2t] * units.kelvin
    q = ds_q[var_q] * units.dimensionless
    z = mpcalc.geopotential_to_height(zs * units('m^2/s^2'))

    # --- Compute MSE using metpy ---
    mse = mpcalc.moist_static_energy(z, T, q)

    mse_ds = mse.to_dataset(name="mse")
    mse_ds.attrs.update({
        "long_name": "Surface moist static energy", "units": "J kg-1"
    })

    return mse_ds

def compute_sat_mse(res, hPa = 500):
    """
    Compute surface moist static energy (MSE) from 2m temperature, specific humidity, and surface geopotential.

    Parameters
    ----------
    res : dict
        Output of `open_era5_day`, containing:
          - res['datasets']['2t']: xarray.Dataset with 2m temperature (K)
          - res['datasets']['q'] : xarray.Dataset with specific humidity (kg/kg)
          - res['zs']            : xarray.DataArray with surface geopotential height (m)

    Returns
    -------
    mse_da : xarray.DataArray
        Surface moist static energy (J/kg) with the same coordinates as the 2m temperature field.
    """

    # --- Extract variables ---
    ds_t  = res["datasets"]["t"][["t","lat", "lon", "time"]]
    ds_z  = res["datasets"]["z"][["z", "lat", "lon", "time"]]

    # Try to find the variable names automatically
    var_t = list(ds_t.data_vars)[0]
    var_z = list(ds_z.data_vars)[0]

    T = ds_t[var_t] * units.kelvin
    z = mpcalc.geopotential_to_height(ds_z[var_z] * units('m^2/s^2'))

    p_sat = mpcalc.saturation_vapor_pressure(T)
    q_sat = 0.622*p_sat/(500* units.hPa)

    # --- Compute MSE using metpy ---
    mse = mpcalc.moist_static_energy(z, T, q_sat)

    mse_ds = mse.to_dataset(name="mse_sat")
    mse_ds.attrs.update({
        "long_name": f"Saturated moist static energy at {hPa} hPa", "units": "J kg-1"
    })
    
    return mse_ds


def compute_temperature_bounds(
    ds,
    T500_var="t",
    ps_var="sp",
    mse_var="mse",
    mse_sat_var="mse_sat",
    p_ref=500e2,
    R=287.0,
    cp=1004.0,
):
    """
    Compute theoretical temperature upper bound and MSE-based temperature bound.

    Parameters
    ----------
    ds : xarray.Dataset
        Input dataset containing pressure, temperature, and MSE variables.
    T500_var : str
        Name of temperature variable at 500 hPa (K).
    ps_var : str
        Name of surface pressure variable (Pa).
    mse_var : str
        Name of moist static energy variable.
    mse_sat_var : str
        Name of saturated moist static energy variable.
    p_ref : float
        Reference pressure in Pa (default: 500 hPa).
    R : float
        Dry air gas constant (J/kg/K).
    cp : float
        Dry air heat capacity (J/kg/K).

    Returns
    -------
    ds : xarray.Dataset
        Dataset with added variables:
        - t_bound
        - t_bound_mse
    """

    ps = ds[ps_var]
    T500 = ds[T500_var]

    # Theoretical upper temperature bound
    ds["t_bound"] = T500 * (ps / p_ref) ** (R / cp)

    # MSE-based temperature bound
    ds["t_bound_mse"] = (-ds[mse_var] + ds[mse_sat_var]) * 1e3 / cp

    return ds


In [1]:
import xarray as xr

ds = xr.open_dataset(
    "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"
)

# Dimensions you want to remove entirely
dims_to_drop = ["depth", "depth_2", "lev", "nhyi", "nhym"]

# Only drop those that actually exist (robust)
dims_to_drop = [d for d in dims_to_drop if d in ds.dims]

ds = ds.drop_dims(dims_to_drop)

# Optional: also drop associated coordinates if they linger
coords_to_drop = [c for c in dims_to_drop if c in ds.coords]
if coords_to_drop:
    ds = ds.drop_vars(coords_to_drop)

# Save cleaned dataset
#out_file = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled_clean.nc"
#ds.to_netcdf(out_file)

In [5]:

ds = xr.open_dataset(
    "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"
)


In [7]:
ds.q

<xarray.DataArray 'q' (time: 7820, lev: 1, lat: 89, lon: 1280)>
[890854400 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 1940-06-01 1940-06-02 ... 2024-08-31
  * lon      (lon) float64 0.0 0.2812 0.5625 0.8438 ... 358.9 359.2 359.4 359.7
  * lat      (lat) float64 64.78 64.5 64.22 63.93 ... 40.89 40.61 40.33 40.05
  * lev      (lev) float64 137.0
    plev     float64 5e+04
Attributes:
    standard_name:     specific_humidity
    long_name:         Specific humidity
    units:             kg kg**-1
    param:             0.1.0
    CDI_grid_type:     gaussian
    CDI_grid_num_LPE:  320
    institution:       ECMWF

In [ ]:
z_s = xr.open_dataset("/work/uc1275/u301827/02_MSE/surface_geopotential_zs.nc")